In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from scraper import fetch_website_contents

ImportError: cannot import name 'fetch_website_contents' from 'scraper' (c:\Learnership\Python projects\AI-Application\.venv\Lib\site-packages\scraper\__init__.py)

In [ ]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=openai_api_key)


In [ ]:
system_message = """
You are an assistant that analyzes the contents of a company website landing page
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
"""

In [ ]:
def stream_gpt(prompt):
    messages = [{"role":"system", "content":system_message}, {"role":"user", "content":prompt}]
    response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages, stream=True)
    result = ""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result

In [ ]:
def stream_brochure(company_name, url):
    yield ""
    prompt = f"Please generate a company brochure for {company_name}. Here is their landing page:\n"
    prompt += fetch_website_contents(url)
    yield from stream_gpt(prompt)

In [ ]:
name_input = gr.Textbox(label="Company name:")
url_input = gr.Textbox(label="Landing page URL including http:// or https://")
message_output = gr.Markdown(label="Response:")

view = gr.Interface(
    fn=stream_brochure,
    title="Brochure Generator", 
    inputs=[name_input, url_input], 
    outputs=[message_output], 
    examples=[
        ["Hugging Face", "https://huggingface.co/"],
        ["DM Academy", "https://www.dm-academy.app/"]
    ], 
    flagging_mode="never"
)

In [ ]:
view.launch(inbrowser=True)